# Dashboard Data Prep

 **Goal:** Build all dataset tables that power the 5 AI/BI dashboards.
 Each dataset is a clean, pre-aggregated Gold table — one per dashboard widget.

 | Dashboard | Tables it uses |
 |-----------|---------------|
 | 1. Complaint Heatmap | `dash_heatmap` |
 | 2. Ward-wise Trends | `dash_ward_trends` |
 | 3. SLA Analytics | `dash_sla_analytics` |
 | 4. Department Performance | `dash_dept_performance` |
 | 5. Escalation Insights | `dash_escalation_insights` |


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.gold")

## 1. Complaint Heatmap Dataset

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dash_heatmap AS
SELECT
    borough_clean                               AS borough,
    complaint_type,
    open_year                                   AS year,
    open_month                                  AS month,
    MAKE_DATE(open_year, open_month, 1)         AS period,
    COUNT(*)                                    AS total_complaints,
    SUM(CASE WHEN is_closed = FALSE
             THEN 1 ELSE 0 END)                 AS unresolved,
    ROUND(AVG(days_open), 1)                    AS avg_days_open,
    SUM(CASE WHEN has_geo = TRUE
             THEN 1 ELSE 0 END)                 AS geo_tagged_count
FROM civicops.silver.complaints
WHERE borough_clean IS NOT NULL
  AND open_year IS NOT NULL
GROUP BY borough_clean, complaint_type, open_year, open_month
""")
print(f"✓ dash_heatmap: {spark.table('civicops.gold.dash_heatmap').count():,} rows")

## 2. Ward-wise Trends Dataset

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dash_ward_trends AS
WITH monthly AS (
    SELECT
        borough_clean                           AS borough,
        open_year                               AS year,
        open_month                              AS month,
        MAKE_DATE(open_year, open_month, 1)     AS period,
        COUNT(*)                                AS complaint_count,
        SUM(CASE WHEN is_closed = FALSE
                 THEN 1 ELSE 0 END)             AS unresolved,
        ROUND(AVG(days_open), 1)                AS avg_days_open
    FROM civicops.silver.complaints
    WHERE borough_clean IS NOT NULL
    GROUP BY borough_clean, open_year, open_month
)
SELECT
    *,
    complaint_count - LAG(complaint_count) OVER (
        PARTITION BY borough ORDER BY year, month
    )                                           AS mom_change,
    ROUND(
        (complaint_count - LAG(complaint_count) OVER (
            PARTITION BY borough ORDER BY year, month
        )) * 100.0 / NULLIF(LAG(complaint_count) OVER (
            PARTITION BY borough ORDER BY year, month
        ), 0), 1
    )                                           AS mom_pct_change
FROM monthly
""")
print(f"✓ dash_ward_trends: {spark.table('civicops.gold.dash_ward_trends').count():,} rows")

## 3. SLA Analytics Dataset

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dash_sla_analytics AS
SELECT
    h.category,
    h.priority_label,
    h.severity_score,
    h.dept_code,
    h.dept_name,
    DATE_TRUNC('month', h.created_at)           AS month,
    COUNT(*)                                    AS total_tickets,
    -- SLA breach tracking from status history
    COUNT(DISTINCT CASE WHEN s.sla_breached = TRUE
                        THEN s.ticket_id END)   AS breached_count,
    ROUND(
        COUNT(DISTINCT CASE WHEN s.sla_breached = TRUE
                            THEN s.ticket_id END) * 100.0 / COUNT(*), 1
    )                                           AS breach_rate_pct,
    -- Resolution tracking from memory
    COUNT(CASE WHEN h.status = 'RESOLVED'
               THEN 1 END)                      AS resolved_count,
    COUNT(CASE WHEN h.status = 'OPEN'
               THEN 1 END)                      AS open_count,
    ROUND(AVG(h.sla_hours), 0)                  AS avg_sla_target_hrs,
    -- SLA health score: 100 = perfect, 0 = all breached
    ROUND(
        (1 - COUNT(DISTINCT CASE WHEN s.sla_breached = TRUE
                                 THEN s.ticket_id END) * 1.0 / COUNT(*)) * 100, 1
    )                                           AS sla_health_score
FROM civicops.memory.complaint_history h
LEFT JOIN civicops.memory.sla_status_history s
    ON h.ticket_id = s.ticket_id
GROUP BY h.category, h.priority_label, h.severity_score,
         h.dept_code, h.dept_name, DATE_TRUNC('month', h.created_at)
""")
print(f"✓ dash_sla_analytics: {spark.table('civicops.gold.dash_sla_analytics').count():,} rows")

## 4. Department Performance Dataset

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dash_dept_performance AS
SELECT
    t.dept_code,
    t.dept_name,
    t.category,
    COUNT(*)                                                AS total_tickets,
    SUM(CASE WHEN t.escalate = TRUE THEN 1 ELSE 0 END)     AS escalated,
    SUM(CASE WHEN t.health_risk = TRUE THEN 1 ELSE 0 END)  AS health_risk_tickets,
    SUM(CASE WHEN t.field_visit_needed = TRUE
             THEN 1 ELSE 0 END)                            AS field_visits_needed,
    ROUND(AVG(t.sla_hours), 0)                             AS avg_sla_hours,
    ROUND(AVG(t.processing_ms) / 1000.0, 2)                AS avg_ai_processing_sec,
    -- From memory: resolution rate
    COUNT(CASE WHEN h.status = 'RESOLVED'
               THEN 1 END)                                 AS resolved_count,
    ROUND(
        COUNT(CASE WHEN h.status = 'RESOLVED' THEN 1 END) * 100.0 / COUNT(*), 1
    )                                                      AS resolution_rate_pct,
    -- Severity distribution
    ROUND(AVG(t.severity_score), 2)                        AS avg_severity,
    SUM(CASE WHEN t.severity_score >= 4 THEN 1 ELSE 0 END) AS critical_high_count,
    MAX(t.created_at)                                      AS last_ticket_at
FROM civicops.gold.processed_tickets t
LEFT JOIN civicops.memory.complaint_history h
    ON t.ticket_id = h.ticket_id
GROUP BY t.dept_code, t.dept_name, t.category
""")
print(f"✓ dash_dept_performance: {spark.table('civicops.gold.dash_dept_performance').count():,} rows")

## 5. Escalation Insights Dataset

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dash_escalation_insights AS
WITH dept_totals AS (
    SELECT dept_code, COUNT(*) AS dept_total_tickets
    FROM civicops.gold.processed_tickets
    GROUP BY dept_code
),
escalations AS (
    SELECT
        e.escalated_from_dept,
        e.escalated_to_dept,
        h.category,
        h.severity_score,
        DATE_TRUNC('month', e.escalated_at)             AS month,
        COUNT(*)                                        AS escalation_count,
        SUM(CASE WHEN h.health_risk = TRUE
                 THEN 1 ELSE 0 END)                     AS health_risk_escalations,
        SUM(CASE WHEN h.infrastructure_risk = TRUE
                 THEN 1 ELSE 0 END)                     AS infra_risk_escalations,
        SUM(CASE WHEN e.resolved_after_escalation = TRUE
                 THEN 1 ELSE 0 END)                     AS resolved_after_escalation,
        ROUND(AVG(e.resolution_time_hrs), 1)            AS avg_resolution_hrs
    FROM civicops.memory.escalation_context e
    JOIN civicops.memory.complaint_history h
        ON e.ticket_id = h.ticket_id
    GROUP BY e.escalated_from_dept, e.escalated_to_dept, h.category,
             h.severity_score, DATE_TRUNC('month', e.escalated_at)
)
SELECT
    esc.*,
    ROUND(esc.escalation_count * 100.0 / NULLIF(dt.dept_total_tickets, 0), 2) AS escalation_rate_pct
FROM escalations esc
LEFT JOIN dept_totals dt
    ON esc.escalated_from_dept = dt.dept_code
""")
print(f"✓ dash_escalation_insights: {spark.table('civicops.gold.dash_escalation_insights').count():,} rows")

## 6. Verify All Dashboard Tables

In [0]:
dash_tables = [
    "civicops.gold.dash_heatmap",
    "civicops.gold.dash_ward_trends",
    "civicops.gold.dash_sla_analytics",
    "civicops.gold.dash_dept_performance",
    "civicops.gold.dash_escalation_insights",
]

print(f"{'Table':<45} {'Rows':>8}")
print("-" * 55)
for t in dash_tables:
    try:
        n = spark.table(t).count()
        print(f"{t:<45} {n:>8,}")
    except Exception as e:
        print(f"{t:<45} ERROR: {e}")